# Studi Kasus Decision Tree: Kapan Pompa Irigasi Sawah Harus Menyala?

**Tujuan.** Menunjukkan cara kerja decision tree dari hitungan manual (entropy dan information gain) sampai tree yang di-training di scikit-learn, dengan satu kasus yang bisa dijalankan siapa pun tanpa file atau akun.

**Cara menjalankan.** Butuh Python 3.9+ dan `numpy`, `pandas`, `scikit-learn`, `matplotlib`. Semua data dibuat di dalam notebook. Jalankan semua cell dari atas ke bawah (*Run All*).

**Struktur**

| Bagian | Isi | Data |
|---|---|---|
| A | Hitung manual entropy dan information gain, lalu cocokkan dengan scikit-learn | Table mini 16 row, ditulis langsung di cell |
| B | Tree pada data lebih realistis: pembuatan data, compression ke event, training, evaluation, pruning | Data sintetis dari simulator sistem irigasi |
| C | Keterbatasan dan latihan | |

**Catatan kejujuran (baca dulu).**
1. Data di Bagian B **sintetis**: dibangkitkan dari rule controller yang saya tulis sendiri, ditambah sensor noise dan intervensi manual acak. Tree yang di-training pada data ini **menemukan kembali rule tersebut**, bukan menemukan pengetahuan baru tentang sawah. Nilai edukasinya ada pada *cara* tree menemukannya dan seberapa baik ia melakukannya di bawah noise.
2. Parameter fisik simulator (laju penguapan, kenaikan kelembaban saat pompa ON, dan seterusnya) adalah **value ilustratif**. Jika Anda ingin klaim "dimodelkan dari data lapangan", estimasi parameter itu dari data nyata Anda dan ganti isi `PARAMS` pada Bagian B.
3. Semua angka pada output dihasilkan saat notebook dijalankan, dengan seed tetap. Hasil bisa sedikit berbeda antar versi numpy/scikit-learn.

In [ ]:
# Make sure dependencies are available: any missing package is installed via pip.
import importlib.util
import subprocess
import sys

PIP_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}

missing = [package for module, package in PIP_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    print("Done. If the import in the next cell still fails, restart the kernel and run the cells again.")
else:
    print("All dependencies are already installed.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import balanced_accuracy_score

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
SEED = 42

---
# Bagian A. Hitung manual pada dataset mini

Scenario: pompa dikendalikan dari tiga condition yang sudah dikategorikan.

| Feature | Value |
|---|---|
| `soil` | dry, moist, wet |
| `rain` | yes, no |
| `water` (tinggi air di tangki) | low, enough |
| **Target** `pump` | ON, OFF |

Rule "sebenarnya": pompa ON jika tanah kering, air cukup, dan tidak hujan. Satu row (nomor 14) sengaja dibuat melanggar rule, mewakili petani yang menyalakan pompa secara manual. Tujuannya agar tree tidak sempurna dan ada yang bisa dibahas.

In [ ]:
mini = pd.DataFrame([
    # soil,   rain,   water,    pump
    ("dry",   "no",   "enough", "ON"),
    ("dry",   "no",   "enough", "ON"),
    ("dry",   "no",   "enough", "ON"),
    ("dry",   "no",   "low",    "OFF"),
    ("dry",   "yes",  "enough", "OFF"),
    ("dry",   "no",   "enough", "ON"),
    ("moist", "no",   "enough", "OFF"),
    ("moist", "no",   "enough", "OFF"),
    ("moist", "yes",  "enough", "OFF"),
    ("moist", "no",   "low",    "OFF"),
    ("wet",   "no",   "enough", "OFF"),
    ("wet",   "yes",  "enough", "OFF"),
    ("dry",   "yes",  "low",    "OFF"),
    ("moist", "no",   "enough", "ON"),    # manual intervention (rule violator)
    ("wet",   "no",   "enough", "OFF"),
    ("dry",   "no",   "enough", "ON"),
], columns=["soil", "rain", "water", "pump"])
mini.index = mini.index + 1
mini

## A1. Entropy dan Gini

Untuk dataset dengan proportion class $p_i$:

$$H = -\sum_i p_i \log_2 p_i \qquad\qquad G = 1 - \sum_i p_i^2$$

Value 0 berarti pure (satu class saja). Untuk dua class, entropy maksimum (1 bit) saat proportion-nya 50:50.

**Information gain** sebuah split = entropy sebelum split dikurangi rata-rata entropy sesudah split (di-weight oleh jumlah row tiap branch).

In [ ]:
def entropy(y):
    p = y.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def gini(y):
    p = y.value_counts(normalize=True)
    return float(1 - (p ** 2).sum())

def binary_information_gain(df, column, value, target="pump"):
    # Binary split: (column == value) versus the rest, which is what CART/scikit-learn does.
    yes = df[df[column] == value]
    no = df[df[column] != value]
    entropy_after = len(yes) / len(df) * entropy(yes[target]) + len(no) / len(df) * entropy(no[target])
    return entropy(df[target]) - entropy_after

n_on = (mini.pump == "ON").sum()
print(f"Rows         : {len(mini)}  (ON = {n_on}, OFF = {len(mini) - n_on})")
print(f"Root entropy : {entropy(mini.pump):.4f} bit")
print(f"Root gini    : {gini(mini.pump):.4f}")

## A2. Memilih split di root

scikit-learn membuat split **binary**, jadi feature dengan 3 value (`soil`) dipecah menjadi 3 candidate pertanyaan yes/no ("apakah `soil = dry`?", dan seterusnya). Kita hitung information gain semua candidate lalu urutkan.

In [ ]:
candidates = []
for column in ["soil", "rain", "water"]:
    for value in mini[column].unique():
        candidates.append({"question": f"{column} = {value}?",
                           "information_gain": binary_information_gain(mini, column, value)})

ig_table = (pd.DataFrame(candidates)
            .sort_values("information_gain", ascending=False)
            .reset_index(drop=True))
ig_table["information_gain"] = ig_table["information_gain"].round(4)
ig_table

In [ ]:
best_split = ig_table.iloc[0]["question"]
column, value = best_split.replace("?", "").split(" = ")
print(f"Best root split: {best_split}\n")

branch_detail = []
for branch, subset in [("YES", mini[mini[column] == value]), ("NO", mini[mini[column] != value])]:
    branch_detail.append({"branch": branch, "n": len(subset),
                          "ON": (subset.pump == "ON").sum(), "OFF": (subset.pump == "OFF").sum(),
                          "entropy": round(entropy(subset.pump), 4)})
branch_detail = pd.DataFrame(branch_detail)
display(branch_detail)

entropy_after = sum(r.n / len(mini) * r.entropy for r in branch_detail.itertuples())
print(f"\nEntropy before  : {entropy(mini.pump):.4f}")
print(f"Entropy after   : {entropy_after:.4f}  (weighted average)")
print(f"Information gain: {entropy(mini.pump) - entropy_after:.4f}")

## A3. Branch kedua

Branch `NO` (soil bukan dry) hampir semuanya OFF, kecuali satu row pelanggar. Branch `YES` masih campuran, jadi kita ulangi prosedur yang sama hanya pada row di branch itu.

In [ ]:
subset = mini[mini[column] == value]
print(f"Subset: {column} = {value}  ({len(subset)} rows, entropy {entropy(subset.pump):.4f})\n")

candidates2 = []
for feature in ["rain", "water"]:
    for feature_value in subset[feature].unique():
        candidates2.append({"question": f"{feature} = {feature_value}?",
                            "information_gain": round(binary_information_gain(subset, feature, feature_value), 4)})
display(pd.DataFrame(candidates2).sort_values("information_gain", ascending=False).reset_index(drop=True))

## A4. Cocokkan dengan scikit-learn

Jika hitungan manual benar, tree scikit-learn dengan `criterion="entropy"` harus memilih split root yang sama dan menghasilkan information gain yang sama.

In [ ]:
X_mini = pd.get_dummies(mini[["soil", "rain", "water"]]).astype(int)
y_mini = mini["pump"]

tree_mini = DecisionTreeClassifier(criterion="entropy", random_state=0).fit(X_mini, y_mini)
t = tree_mini.tree_

# Root information gain according to scikit-learn, computed from the impurity of each node.
left, right = t.children_left[0], t.children_right[0]
ig_sklearn = t.impurity[0] - (t.n_node_samples[left] / t.n_node_samples[0] * t.impurity[left]
                              + t.n_node_samples[right] / t.n_node_samples[0] * t.impurity[right])

print("Root split (scikit-learn)       :", X_mini.columns[t.feature[0]])
print(f"Root information gain           : {ig_sklearn:.4f}")
print(f"Hand-computed information gain  : {ig_table.iloc[0]['information_gain']:.4f}")
assert abs(ig_sklearn - ig_table.iloc[0]["information_gain"]) < 1e-3, "hand computation does not match"
print("\nThey match.\n")

print(export_text(tree_mini, feature_names=list(X_mini.columns)))

In [ ]:
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import to_rgb

def plot_clean_tree(clf, feature_names, title, unit="time"):
    """Draw the tree top-down: one-line conditions, class-coloured leaves, no filler."""
    t = clf.tree_
    classes = list(clf.classes_)
    palette = {"ON": "#f4a261", "OFF": "#90caf9"}

    # Depth becomes the x axis; leaves are stacked on the y axis in order.
    depth_of, y_of = {}, {}
    next_leaf = [0]
    def place(node, depth):
        left, right = t.children_left[node], t.children_right[node]
        if left == right:
            depth_of[node], y_of[node] = depth, -next_leaf[0]
            next_leaf[0] += 1
        else:
            place(left, depth + 1)
            place(right, depth + 1)
            y_of[node] = (y_of[left] + y_of[right]) / 2
            depth_of[node] = depth
    place(0, 0)

    n_leaves = max(next_leaf[0], 1)
    max_depth = max(depth_of.values())
    step_x, box_w, box_h = 2.3, 1.5, 0.52
    fig, ax = plt.subplots(figsize=(step_x * (max_depth + 1) + 0.8, 0.72 * n_leaves + 1.0))

    def node_text(node):
        left, right = t.children_left[node], t.children_right[node]
        if left == right:                          # leaf: predicted class + share of the data
            value = t.value[node][0] / t.value[node][0].sum()
            class_label = classes[int(value.argmax())]
            share = 100 * t.weighted_n_node_samples[node] / t.weighted_n_node_samples[0]
            return f"{class_label}\n{share:.0f}% of {unit}", class_label, value.max()
        feature, value = feature_names[t.feature[node]].split("_", 1)   # split: one yes/no question
        return f"{feature} = {value}?", None, None

    for node in range(t.node_count):
        text, class_label, purity = node_text(node)
        x, y = depth_of[node] * step_x, y_of[node]
        left, right = t.children_left[node], t.children_right[node]
        x_right = x + box_w / 2

        if left != right:
            for child, answer in [(left, "no"), (right, "yes")]:
                x_child = depth_of[child] * step_x - box_w / 2
                ax.plot([x_right, x_child], [y, y_of[child]], color="#bdbdbd", lw=1.1, zorder=1)
                ax.text(x_right + (x_child - x_right) * 0.55, y + (y_of[child] - y) * 0.55, answer,
                        ha="center", va="center", fontsize=8, color="#616161",
                        bbox=dict(facecolor="white", edgecolor="none", pad=0.3), zorder=3)

        # Leaf colour = predicted class, stronger when the leaf is purer.
        face = "#f5f5f5" if class_label is None else tuple(1 - purity * (1 - np.array(to_rgb(palette[class_label]))))
        ax.add_patch(FancyBboxPatch((x - box_w / 2, y - box_h / 2), box_w, box_h,
                                    boxstyle="round,pad=0.02,rounding_size=0.1",
                                    facecolor=face, edgecolor="#9e9e9e", lw=0.9, zorder=2))
        ax.text(x, y, text, ha="center", va="center", fontsize=10, color="#212121",
                linespacing=1.35, zorder=4)

    ax.set_xlim(-box_w / 2 - 0.2, max_depth * step_x + box_w / 2 + 0.2)
    ax.set_ylim(min(y_of.values()) - 0.55, max(y_of.values()) + 0.55)
    ax.axis("off")
    ax.set_title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.show()

plot_clean_tree(tree_mini, list(X_mini.columns), "Mini dataset tree", unit="rows")

**Catatan.**
- Row 14 (pelanggar) memiliki value feature identik dengan row 7 dan 8 (moist, no rain, enough) tetapi label berbeda. Tree **tidak bisa** memisahkannya dengan feature yang ada, jadi leaf itu tetap campuran dan diprediksi sebagai class mayoritas (OFF). Ini contoh *label noise*: error yang tidak bisa dihilangkan oleh tree sedalam apa pun.
- Beberapa branch pada tree tanpa batas depth berakhir di leaf yang sama-sama OFF. Itu terjadi karena tree tetap memecah node yang belum pure walau prediction-nya tidak berubah. Pada Bagian B kita membatasi depth agar tree tidak membuang split seperti ini.
- Pada `export_text` di atas, `soil_dry <= 0.50` berarti jawaban **No** (value 0), `> 0.50` berarti **Yes**.
- Algorithm klasik ID3/C4.5 memakai split *multiway* (satu branch per value). scikit-learn (CART) memakai split binary, sehingga angka information gain bisa berbeda dari buku teks yang memakai ID3.

---
# Bagian B. Tree pada data yang lebih realistis

Data mini terlalu bersih. Di sini kita buat data sensor sintetis selama 21 hari dengan resolusi 1 menit, lalu memperlakukannya seperti data lapangan: dikategorikan, dikompresi menjadi event, dan di-evaluate secara temporal.

## B1. Parameter simulator

Semua angka ada di satu tempat agar mudah diganti. **Value di bawah ilustratif.** Cara mengestimasi dari data nyata ada di comment kanan.

In [ ]:
PARAMS = {
    # --- environment ---
    "days": 21,
    "initial_moisture": 50.0,          # % (relative water content)
    "initial_water_level": 8.0,        # arbitrary unit (e.g. dm) in the tank
    "daytime_evaporation": 0.05,       # %/min, 08:00-16:00  <- moisture slope when pump OFF, no rain, daytime
    "night_evaporation": 0.02,         # %/min, other hours    <- same, at night
    "pump_moisture_gain": 0.25,        # %/min while pump ON   <- moisture slope while pump ON
    "sensor_lag": 20,                  # minutes: the root-zone sensor "sees" water after it seeps in <- lag between pump ON and the rise in reading
    "rain_moisture_gain": 0.25,        # %/min while raining   <- moisture slope while raining
    "rain_probability": 0.0004,        # chance of rain starting per minute
    "rain_duration": (30, 90),         # minutes (min, max)
    "water_drop_idle": 0.001,          # water level/min (seepage, evaporation)
    "water_drop_pump": 0.005,          # extra water level/min while pump ON
    "tank_refill_hour": 6,             # the farmer refills the tank at 06:00 if it is below the threshold
    "tank_refill_level": 10.0,
    "sensor_noise": 0.7,               # std of moisture sensor noise (%)  <- std of successive differences on stable segments

    # --- pump control rules (the ones the tree should "discover") ---
    "turn_on_threshold": 40.0,         # pump turns on if the raw reading < 40 %
    "minimum_water_level": 5.0,        # and water level >= 5
    "forbidden_hours": (12, 18),       # pump may not run 12:00-17:59 (avoid daytime evaporation)
    "min_on_minutes": 10,              # relay guard: once ON, it stays on at least 10 minutes
    "override_probability": 0.001,     # chance of manual intervention per minute
    "override_duration": (10, 30),     # minutes

    # --- feature binning ---
    "dry_threshold": 42.0,             # soil: <=42 dry, <=55 moist, else wet (slightly above the pump threshold of 40)
    "wet_threshold": 55.0,
    "debounce_window": 15,             # rolling median of 15 minutes (past data only, so it is usable for real time)
}
pd.Series({k: str(v) for k, v in PARAMS.items()}, name="value").to_frame()

## B2. Simulator

Setiap menit simulator memperbarui kelembaban tanah dan tinggi air, lalu pompa memutuskan hidup/mati berdasarkan **pembacaan sensor mentah yang berisik dan tertinggal** (`sensor_lag`: air baru terbaca di root zone setelah beberapa menit meresap). Feature yang nanti dilihat tree dihitung dari pembacaan yang sudah dihaluskan, jadi ada ketidakcocokan kecil antara "yang memutuskan" dan "yang dilihat tree". Ketidakcocokan ini realistis dan menjadi sumber error tree.

Sensor lag juga punya fungsi didaktik. Tanpanya, pompa yang menyala langsung menaikkan pembacaan melewati threshold, sehingga pompa ON hampir selalu terjadi pada category `moist` dan tree dengan feature snapshot tidak punya sinyal untuk dipelajari (controller dengan hysteresis tidak bisa dibaca dari satu state saja; lihat Bagian C).

Dua feature tambahan sengaja dibuat:
- `time` (night/morning/noon/afternoon): **relevan**, karena pompa dilarang menyala pada bin `noon` (12-17), sedangkan `afternoon` (18-23) tidak dilarang.
- `wind` (north/south/west): **tidak relevan**, hanya proxy. Tree yang sehat seharusnya mengabaikannya.

In [ ]:
def simulate(params, seed=SEED):
    rng = np.random.default_rng(seed)
    n = params["days"] * 24 * 60
    timestamps = pd.date_range("2025-01-01", periods=n, freq="min")

    moisture, water_level = params["initial_moisture"], params["initial_water_level"]
    pump_on, rain_remaining, override_remaining, min_on_remaining = False, 0, 0, 0
    override_state, wind = False, "north"
    moisture_history = []          # true moisture per minute, used to model the sensor lag
    rows = []

    for ts in timestamps:
        # weather
        if rain_remaining == 0 and rng.random() < params["rain_probability"]:
            rain_remaining = int(rng.integers(*params["rain_duration"]))
        raining = rain_remaining > 0
        rain_remaining = max(0, rain_remaining - 1)
        if ts.minute == 0 and ts.hour % 6 == 0:
            wind = str(rng.choice(["north", "south", "west"]))

        # physics
        evaporation = params["daytime_evaporation"] if 8 <= ts.hour < 16 else params["night_evaporation"]
        moisture = float(np.clip(moisture - evaporation + (params["rain_moisture_gain"] if raining else 0)
                                 + (params["pump_moisture_gain"] if pump_on else 0), 0, 100))
        water_level = float(np.clip(water_level - params["water_drop_idle"] - (params["water_drop_pump"] if pump_on else 0), 0, 15))
        if ts.hour == params["tank_refill_hour"] and ts.minute == 0 and water_level < params["minimum_water_level"]:
            water_level = params["tank_refill_level"]

        # sensor and pump decision
        moisture_history.append(moisture)
        sensor_moisture = moisture_history[max(0, len(moisture_history) - 1 - params["sensor_lag"])]   # sensor lags behind
        raw_reading = sensor_moisture + rng.normal(0, params["sensor_noise"])
        time_allowed = not (params["forbidden_hours"][0] <= ts.hour < params["forbidden_hours"][1])
        automatic_on = (raw_reading < params["turn_on_threshold"]) and (water_level >= params["minimum_water_level"]) and (not raining) and time_allowed

        if override_remaining == 0 and rng.random() < params["override_probability"]:
            override_remaining = int(rng.integers(*params["override_duration"]))
            override_state = not automatic_on
        if override_remaining > 0:
            pump_on, override_remaining = override_state, override_remaining - 1
        elif min_on_remaining > 0:
            pump_on, min_on_remaining = True, min_on_remaining - 1
        else:
            pump_on = automatic_on
            if automatic_on:
                min_on_remaining = params["min_on_minutes"] - 1

        rows.append((ts, raw_reading, water_level, raining, pump_on, wind))

    return pd.DataFrame(rows, columns=["ts", "raw_moisture", "water_level", "rain", "pump", "wind"])

raw = simulate(PARAMS)
print(f"{len(raw):,} rows (1 row = 1 minute, {PARAMS['days']} days)")
raw.head()

### Seperti apa datanya?

Plot tiga hari: garis biru adalah kelembaban (sudah dihaluskan), area oranye adalah saat pompa ON, garis putus-putus adalah threshold 40%, titik biru di bawah adalah hujan. Dua hal terlihat: (1) pada jam 12-18 kelembaban turun jauh di bawah threshold tetapi pompa baru menyala setelah jam 18 (larangan jam siang); (2) saat tinggi air di bawah garis putus-putus pada panel bawah (tangki rendah), pompa tidak menyala meski tanah kering.

In [ ]:
df = raw.copy()
df["moisture"] = df["raw_moisture"].rolling(PARAMS["debounce_window"], min_periods=1).median()

start = pd.Timestamp("2025-01-08")
window = df[(df.ts >= start) & (df.ts < start + pd.Timedelta(days=3))]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
ax1.plot(window.ts, window.moisture, color="tab:blue", lw=1.3, label="soil moisture (15-min median)")
ax1.axhline(PARAMS["turn_on_threshold"], color="gray", ls="--", lw=1, label=f"ON threshold ({PARAMS['turn_on_threshold']:.0f}%)")
ax1.fill_between(window.ts, 0, 1, where=window.pump, transform=ax1.get_xaxis_transform(),
                 color="tab:orange", alpha=0.35, label="pump ON")
ax1.fill_between(window.ts, 0, 1, where=window.rain, transform=ax1.get_xaxis_transform(),
                 color="tab:cyan", alpha=0.15, label="rain")
ax1.set_ylabel("moisture (%)"); ax1.legend(loc="upper right", fontsize=8, ncol=2)
ax1.set_title("Three days of simulated data")

ax2.plot(window.ts, window.water_level, color="tab:green", lw=1.3)
ax2.axhline(PARAMS["minimum_water_level"], color="gray", ls="--", lw=1)
ax2.set_ylabel("water level"); ax2.set_xlabel("time")
plt.tight_layout(); plt.show()

## B3. Dari angka ke category, lalu ke event

Langkah ini meniru rencana awal: data sensor per menit diubah menjadi category, lalu hanya perubahan category yang disimpan.

1. **Categorization.** `soil` dari kelembaban terhaluskan, `water` dari tinggi air, `time` dari jam.
2. **Compression.** Row baru dibuat hanya saat salah satu category (termasuk target `pump`) berubah.
3. **Duration.** Lama tiap event disimpan dalam menit dan dipakai sebagai **sample weight**, supaya state yang berlangsung 8 jam tidak dihitung sama dengan state yang berlangsung 3 menit.

Duration **tidak** dipakai sebagai feature: duration baru diketahui setelah event selesai, sehingga memakainya untuk memprediksi adalah information leakage (*leakage*).

In [ ]:
df["soil"] = pd.cut(df["moisture"], [-np.inf, PARAMS["dry_threshold"], PARAMS["wet_threshold"], np.inf],
                    labels=["dry", "moist", "wet"]).astype(str)
df["water"]  = np.where(df["water_level"] < PARAMS["minimum_water_level"], "low", "enough")
df["time"]   = pd.cut(df["ts"].dt.hour, [-1, 5, 11, 17, 23],
                      labels=["night", "morning", "noon", "afternoon"]).astype(str)
df["rain"]   = df["rain"].map({True: "yes", False: "no"})
df["pump"]   = df["pump"].map({True: "ON", False: "OFF"})

FEATURES = ["soil", "water", "rain", "time", "wind"]
TARGET = "pump"
category_cols = FEATURES + [TARGET]

changed = (df[category_cols] != df[category_cols].shift()).any(axis=1)
events = df.loc[changed, ["ts"] + category_cols].copy()
events["duration_min"] = (events["ts"].shift(-1) - events["ts"]).dt.total_seconds() / 60
events = events.dropna(subset=["duration_min"]).reset_index(drop=True)   # the last event has no end

print(f"Before compression : {len(df):>7,} rows")
print(f"After compression  : {len(events):>7,} events  (ratio {len(df)/len(events):.1f}x)\n")
events.head(12)

In [ ]:
summary = events.groupby(TARGET).agg(events=("duration_min", "size"),
                                     total_minutes=("duration_min", "sum"),
                                     median_minutes=("duration_min", "median"))
summary["time_share_%"] = (100 * summary["total_minutes"] / summary["total_minutes"].sum()).round(1)
summary["event_share_%"] = (100 * summary["events"] / summary["events"].sum()).round(1)
print("Target classes, by number of events versus time spent:")
summary

Perhatikan selisih antara `event_share_%` dan `time_share_%`. Pompa ON adalah bagian kecil dari waktu tetapi bagian lebih besar dari event. Ini alasan pemilihan weight (duration atau tidak) mengubah apa yang dianggap "penting" oleh tree, dan evaluation di bawah memakai balanced accuracy agar class ON yang jarang tidak tertutup oleh class OFF.

## B4. Split temporal dan training

Split acak akan menyebabkan information leakage karena event yang bersebelahan sangat mirip. Kita pakai **70% event pertama untuk training, 30% terakhir untuk test**.

Feature categorical diubah ke bentuk binary (*one-hot*) karena scikit-learn hanya menerima angka.

In [ ]:
X = pd.get_dummies(events[FEATURES]).astype(int)
y = events[TARGET]
w = events["duration_min"]

k = int(len(events) * 0.7)
X_train, X_test = X.iloc[:k], X.iloc[k:]
y_train, y_test = y.iloc[:k], y.iloc[k:]
w_train, w_test = w.iloc[:k], w.iloc[k:]

print(f"Train: {len(X_train)} events, {events.ts.iloc[0]:%d %b} to {events.ts.iloc[k-1]:%d %b}")
print(f"Test : {len(X_test)} events, {events.ts.iloc[k]:%d %b} to {events.ts.iloc[-1]:%d %b}")
print(f"ON share (by time) -> train {w_train[y_train=='ON'].sum()/w_train.sum():.1%}, test {w_test[y_test=='ON'].sum()/w_test.sum():.1%}")

In [ ]:
tree = DecisionTreeClassifier(criterion="entropy", max_depth=4, random_state=0)   # initial tree; the depth is revisited in B7
tree.fit(X_train, y_train, sample_weight=w_train)

feature_names = list(X.columns)
print(f"Initial tree -> depth: {tree.get_depth()}, leaves: {tree.get_n_leaves()}")

## B5. Membaca tree sebagai rule

Daftar berikut mengubah setiap leaf menjadi kalimat IF-THEN, diurutkan dari leaf yang mencakup waktu terlama. Condition berulang pada satu feature diringkas (mis. "bukan noon DAN bukan morning" menjadi `time ∈ {night, afternoon}`). `correct` adalah proportion waktu di leaf tersebut yang class-nya sesuai prediction (leaf purity).

**Cara membaca gambar tree:** kotak internal berisi satu condition (mis. `soil = dry?`) dengan label cabang `no`/`yes`. Kotak daun berisi class prediksi dan share data di leaf itu (di-weight duration), dan warnanya mengikuti class; makin pekat berarti leaf makin pure.

Ini **initial tree** dengan `max_depth=4`, yang ditetapkan sebelum kita tahu depth terbaik. Pemilihan depth dilakukan di B7.

In [ ]:
def print_rules(clf, feature_names):
    possible = {}                                 # feature -> list of possible values
    for name in feature_names:
        feature, value = name.split("_", 1); possible.setdefault(feature, []).append(value)

    t = clf.tree_
    leaves = []
    def walk(node, equals, excludes):
        if t.children_left[node] == t.children_right[node]:          # leaf
            p = t.value[node][0]; p = p / p.sum()
            conditions = []
            for feature in possible:
                if feature in equals:
                    conditions.append(f"{feature} = {equals[feature]}")
                elif excludes.get(feature):
                    remaining = [v for v in possible[feature] if v not in excludes[feature]]
                    conditions.append(f"{feature} = {remaining[0]}" if len(remaining) == 1 else f"{feature} in {{{', '.join(remaining)}}}")
            leaves.append((t.weighted_n_node_samples[node], conditions,
                           clf.classes_[p.argmax()], p.max(), int(t.n_node_samples[node])))
            return
        feature, value = feature_names[t.feature[node]].split("_", 1)
        excluded_next = {key: list(values) for key, values in excludes.items()}; excluded_next.setdefault(feature, []).append(value)
        walk(t.children_left[node], dict(equals), excluded_next)                  # answer No
        walk(t.children_right[node], {**equals, feature: value}, {key: list(values) for key, values in excludes.items()})   # answer Yes
    walk(0, {}, {})

    for weight, conditions, class_label, purity, n in sorted(leaves, key=lambda d: -d[0]):
        condition_text = " AND ".join(conditions) if conditions else "(all conditions)"
        print(f"IF {condition_text}")
        print(f"   THEN pump = {class_label:<3} | {n:>4} events, {weight:>9,.0f} minutes, correct {purity:.0%}\n")

print_rules(tree, feature_names)

In [ ]:
plot_clean_tree(tree, feature_names, "Initial tree (max_depth=4)")

## B6. Evaluation initial tree pada test data

Tiga metric dipakai, semuanya **di-weight duration** agar mencerminkan waktu nyata, bukan jumlah event:
- **Accuracy**: proportion waktu di mana prediction benar.
- **Balanced accuracy**: rata-rata *recall* class ON dan OFF. Tidak tertipu oleh class OFF yang dominan.
- **Precision dan recall class ON**: dari waktu yang diprediksi ON, berapa yang benar-benar ON (precision); dari waktu yang benar-benar ON, berapa yang terdeteksi (recall).

Pembanding: **majority baseline** yang selalu menebak class yang paling lama pada training data. Tree baru bermakna jika mengalahkan baseline ini. Perhatikan bahwa baseline sudah mencapai accuracy tinggi hanya karena pompa lebih sering OFF; itu sebabnya accuracy biasa tidak cukup.

In [ ]:
def metrics(y_true, y_pred, weights):
    actual_on = (y_true == "ON"); predicted_on = (y_pred == "ON")
    tp = weights[actual_on & predicted_on].sum(); fp = weights[~actual_on & predicted_on].sum(); fn = weights[actual_on & ~predicted_on].sum()
    return {
        "accuracy": np.average(y_true == y_pred, weights=weights),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred, sample_weight=weights),
        "precision_ON": tp / (tp + fp) if (tp + fp) else np.nan,
        "recall_ON": tp / (tp + fn) if (tp + fn) else np.nan,
    }

def confusion_matrix(y_true, y_pred, weights):
    cm = (pd.DataFrame({"actual": y_true, "predicted": y_pred, "minutes": weights})
            .pivot_table(index="actual", columns="predicted", values="minutes", aggfunc="sum", fill_value=0)
            .reindex(index=["OFF", "ON"], columns=["OFF", "ON"], fill_value=0))
    cm.index = ["Actual OFF", "Actual ON"]; cm.columns = ["Predicted OFF", "Predicted ON"]
    return cm.round(0).astype(int)

majority_class = y_train.groupby(y_train).apply(lambda s: w_train[s.index].sum()).idxmax()
pred_baseline = pd.Series(majority_class, index=y_test.index)
pred_test = pd.Series(tree.predict(X_test), index=y_test.index)

results = pd.DataFrame({
    f"baseline (always {majority_class}), test": metrics(y_test, pred_baseline, w_test),
    "initial tree, train": metrics(y_train, pd.Series(tree.predict(X_train), index=y_train.index), w_train),
    "initial tree, test": metrics(y_test, pred_test, w_test),
}).T.round(3)
results

In [ ]:
print("Confusion matrix of the initial tree on the test data, in MINUTES (not event counts):")
confusion_matrix(y_test, pred_test, w_test)

## B7. Pruning: seberapa dalam tree sebaiknya?

Tree yang lebih dalam bisa menghafal noise (overfitting). Kita bandingkan beberapa value `max_depth` dan memilih tree **paling sederhana yang hasilnya tidak jauh dari yang terbaik** pada test data (toleransi 0,01 pada balanced accuracy).

Catatan: memilih depth berdasarkan test data lalu melaporkan score test data yang sama sedikit optimistis. Pada studi serius gunakan cross-validation berurutan (`TimeSeriesSplit`) atau validation set terpisah.

In [ ]:
rows = []
for depth in [1, 2, 3, 4, 5, 6, 8, None]:
    clf = DecisionTreeClassifier(criterion="entropy", max_depth=depth, random_state=0).fit(X_train, y_train, sample_weight=w_train)
    rows.append({
        "max_depth": "no limit" if depth is None else depth,
        "leaves": clf.get_n_leaves(),
        "train_balanced_accuracy": balanced_accuracy_score(y_train, clf.predict(X_train), sample_weight=w_train),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, clf.predict(X_test), sample_weight=w_test),
    })
depth_table = pd.DataFrame(rows)

best_test = depth_table["test_balanced_accuracy"].max()
selected = depth_table[depth_table["test_balanced_accuracy"] >= best_test - 0.01].iloc[0]
depth_table["chosen"] = np.where(depth_table["max_depth"].astype(str) == str(selected["max_depth"]), "<--", "")
display(depth_table.round(3))
print(f"Simplest tree within 0.01 of the best: max_depth = {selected['max_depth']} ({int(selected['leaves'])} leaves)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(depth_table["leaves"], depth_table["train_balanced_accuracy"], "o-", label="train")
ax.plot(depth_table["leaves"], depth_table["test_balanced_accuracy"], "s-", label="test")
ax.set_xlabel("number of leaves"); ax.set_ylabel("balanced accuracy")
ax.set_title("Tree complexity vs performance"); ax.legend()
plt.tight_layout(); plt.show()

**Cara membaca (pada run bawaan).**
- Tree dengan satu atau dua level belum cukup: satu pertanyaan tidak mampu memisahkan "ON" dari "OFF", karena pompa ON membutuhkan *kombinasi* tanah kering, air cukup, dan waktu yang diizinkan. Balanced accuracy-nya sama dengan menebak mayoritas.
- Pada depth 3 kombinasi itu tertangkap. Menambah depth **tidak memperbaiki** score training maupun test, bahkan menurunkannya. Ini bukan overfitting klasik (yang biasanya menaikkan score training dan menurunkan score test). Penjelasan yang paling masuk akal, tetapi tidak di-test terpisah di sini: criterion entropy mengejar leaf purity, bukan balanced accuracy, sehingga split lanjutan memecah leaf ON yang bercampur menjadi potongan kecil yang masing-masing mayoritas OFF, dan prediction ON hilang.
- Pelajaran praktisnya: pilih depth dengan metric yang sesuai tujuan, bukan hanya mengandalkan criterion bawaan tree.

## B8. Final tree: rule, kinerja, dan feature yang dipakai

Tree di-training ulang dengan depth terpilih dari B7.

In [ ]:
final_depth = None if str(selected["max_depth"]) == "no limit" else int(selected["max_depth"])
tree_final = DecisionTreeClassifier(criterion="entropy", max_depth=final_depth, random_state=0)
tree_final.fit(X_train, y_train, sample_weight=w_train)

print(f"Final tree: max_depth = {final_depth}, {tree_final.get_n_leaves()} leaves\n")
print_rules(tree_final, feature_names)

In [ ]:
plot_clean_tree(tree_final, feature_names, f"Final tree (max_depth={final_depth})")

In [ ]:
pred_final = pd.Series(tree_final.predict(X_test), index=y_test.index)
comparison = pd.DataFrame({
    f"baseline (always {majority_class})": metrics(y_test, pred_baseline, w_test),
    "initial tree (depth 4)": metrics(y_test, pred_test, w_test),
    f"final tree (depth {final_depth})": metrics(y_test, pred_final, w_test),
}).T.round(3)
print("All on TEST data, weighted by duration:")
display(comparison)
print("\nConfusion matrix of the final tree (minutes):")
confusion_matrix(y_test, pred_final, w_test)

**Cara membaca (pada run bawaan).** Final tree punya accuracy biasa yang lebih rendah daripada initial tree, tetapi balanced accuracy dan recall ON lebih tinggi, dengan precision ON yang lebih rendah. Artinya final tree lebih sering menangkap waktu pompa ON dengan mengorbankan lebih banyak "false alarm" ON. Model mana yang lebih baik bergantung pada biaya kesalahan yang Anda pilih (menebak ON padahal OFF vs melewatkan ON), jadi tentukan metric sebelum melihat hasil.

### Feature mana yang dipakai?

Feature importance dijumlahkan per feature asli (bukan per column one-hot). `wind` adalah proxy feature yang tidak berpengaruh pada rule pompa; value-nya yang mendekati nol menandakan tree tidak tertipu noise.

In [ ]:
importances = pd.Series(tree_final.feature_importances_, index=feature_names)
importances_by_feature = importances.groupby(lambda c: c.split("_", 1)[0]).sum().reindex(FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.2))
importances_by_feature.plot.barh(ax=ax, color="tab:blue")
ax.set_xlabel("share of entropy decrease (total = 1)")
ax.set_title("Feature importance, final tree")
for i, v in enumerate(importances_by_feature.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## B9. Mencoba prediction pada scenario baru

Beberapa situasi hipotetis. Column `true_rule` adalah keputusan menurut rule simulator, sebagai sanity check. Row yang berbeda antara tree dan true rule adalah bagian yang paling menarik dibahas.

In [ ]:
scenarios = pd.DataFrame([
    ("dry",   "enough", "no",  "night",     "north"),
    ("dry",   "enough", "no",  "afternoon", "west"),
    ("dry",   "enough", "no",  "noon",      "west"),
    ("dry",   "low",    "no",  "afternoon", "north"),
    ("dry",   "enough", "yes", "afternoon", "south"),
    ("moist", "enough", "no",  "morning",   "north"),
    ("wet",   "enough", "no",  "night",     "west"),
], columns=FEATURES)

def one_hot_encode(cat_df):
    encoded = pd.DataFrame(0, index=cat_df.index, columns=X.columns)
    for feature in FEATURES:
        for i, value in cat_df[feature].items():
            col = f"{feature}_{value}"
            if col in encoded.columns: encoded.loc[i, col] = 1
    return encoded

X_scenarios = one_hot_encode(scenarios)
scenarios["tree_prediction"] = tree_final.predict(X_scenarios)
scenarios["proba_ON"] = tree_final.predict_proba(X_scenarios)[:, list(tree_final.classes_).index("ON")].round(2)
scenarios["true_rule"] = np.where((scenarios.soil == "dry") & (scenarios.water == "enough")
                                  & (scenarios.rain == "no") & (scenarios.time != "noon"), "ON", "OFF")
scenarios

**Cara membaca selisihnya.** Selisih antara tree dan true rule bisa berasal dari dua hal yang berbeda, dan keduanya pelajaran penting:
- **Tree tidak memakai feature itu.** Rain jarang terjadi (sebagian kecil waktu), sehingga split berdasarkan `rain` menurunkan entropy lebih sedikit daripada split lain. Tree dangkal tidak "tahu" rule rain, padahal rule itu ada. Tree hanya belajar apa yang cukup sering muncul di data.
- **Category terlalu lebar.** `soil = dry` mencakup kelembaban sampai 42%, sedangkan pompa baru menyala di bawah 40%. Pada rentang 40-42% pompa OFF, sehingga probability ON pada category `dry` tidak pernah mendekati 100%. Tree hanya bisa melaporkan probability itu, bukan memperbaikinya.

Perilaku tepatnya bergantung pada data hasil simulasi (di sini `SEED = 42`).

In [ ]:
top_features = importances_by_feature.sort_values(ascending=False)
final_metrics = metrics(y_test, pred_final, w_test)
print("AUTOMATIC SUMMARY")
print("-----------------")
print(f"- Data    : {len(df):,} minute rows -> {len(events)} events ({len(df)/len(events):.0f}x more compact).")
print(f"- Tree    : max_depth {final_depth}, {tree_final.get_n_leaves()} leaves (selected in B7).")
print(f"- Features: most influential = {top_features.index[0]} ({top_features.iloc[0]:.2f}), "
      f"then {top_features.index[1]} ({top_features.iloc[1]:.2f}); proxy feature 'wind' = {importances_by_feature['wind']:.3f}.")
print(f"- Test    : balanced accuracy {final_metrics['balanced_accuracy']:.1%} vs baseline {comparison.iloc[0]['balanced_accuracy']:.1%}.")
print(f"- ON class: precision {final_metrics['precision_ON']:.1%}, recall {final_metrics['recall_ON']:.1%} (duration-weighted).")

---
# Bagian C. Keterbatasan dan latihan

## Keterbatasan yang perlu diakui

1. **Sirkularitas.** Target dihasilkan oleh rule yang kita tulis, sehingga "keberhasilan" tree sebagian besar berarti ia berhasil membaca ulang rule itu. Pada sistem nyata, rule controller belum tentu diketahui, dan di situlah tree bermanfaat sebagai alat penjelas.
2. **Parameter ilustratif.** Tanpa estimasi dari data lapangan, angka seperti laju penguapan tidak boleh diklaim mewakili sawah tertentu.
3. **Hysteresis dan memory.** Controller nyata sering bergantung pada riwayat (misalnya pompa tetap ON sampai tanah mencapai threshold atas). Feature snapshot seperti di sini tidak menangkap riwayat, dan pada simulator ini sinyalnya baru bisa dibaca karena sengaja ditambahkan `sensor_lag`. Trend feature atau status pompa sebelumnya bisa membantu, dengan risiko menjadi proxy dari target itu sendiri (trend kelembaban adalah *akibat* pompa, bukan penyebab).
4. **Threshold category arbitrer.** Batas `dry`/`moist`/`wet` ditetapkan di awal. Jika dipilih sambil melihat target, hasil menjadi bias.
5. **Satu split temporal.** Evaluation memakai satu test split saja, dan ukuran test kecil. Variasi antar-seed belum diukur.
6. **Duration weight adalah pilihan.** Dengan atau tanpa weight, tree bisa berbeda. Tidak ada pilihan yang netral.
7. **Single decision tree.** Cukup untuk menjelaskan, tetapi biasanya kalah stabil dibanding random forest atau gradient boosting pada data nyata.

## Latihan untuk pembaca

1. Ubah `SEED` dan jalankan ulang. Apakah feature dominan dan depth terpilih berubah?
2. Naikkan `sensor_noise` ke 4. Bagaimana accuracy test dan jumlah event berubah?
3. Setel `min_on_minutes` ke 1. Apa yang terjadi pada jumlah event setelah kompresi, dan mengapa?
4. Hapus weight (`sample_weight`) dari training. Bandingkan rule yang dihasilkan.
5. Setel `forbidden_hours` ke `(0, 0)` (tanpa larangan). Apakah `time` hilang dari tree?
6. Ganti `criterion="entropy"` dengan `"gini"`. Seberapa sering hasilnya berbeda?
7. Estimasi parameter `PARAMS` dari data sensor Anda sendiri, lalu bandingkan proportion waktu pompa ON dan distribusi duration antara data nyata dan sintetis.